In [1]:
#!pip install --upgrade seaborn #Upgrade

# Data Preparation

In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import warnings
def ignore_warn(*args, **kwargs):
    pass

warnings.warn = ignore_warn #ignore annoying warning (from sklearn and seaborn)

Train = pd.read_csv('../input/house-prices-advanced-regression-techniques/train.csv')
Train = Train.set_index('Id')
Test = pd.read_csv('../input/house-prices-advanced-regression-techniques/test.csv')
Test = Test.set_index('Id')

preprocessing_path = '../input/house-price-eda'

train_scale = pd.read_csv(f'{preprocessing_path}/train_scaled.csv')
train_scale = train_scale.set_index('Id')
test_scale = pd.read_csv(f'{preprocessing_path}/test_scaled.csv')
try:
    test_scale = test_scale.rename(columns={'Unnamed: 0':'Id'})
except:
    pass
test_scale = test_scale.set_index('Id')

Train  = Train.loc[train_scale.index]

import glob

listing = glob.glob(f'{preprocessing_path}/*_one_hot_pickle')

import pickle
#Preprocess train data
for model_file in listing:
    col = model_file.split('_one_')[0]

    col = col.split('/')[-1]
    enc = pickle.load(open(model_file, 'rb'))
    new_cols = pd.DataFrame(enc.transform(Train[[col]]).toarray(),
                    columns=f'{col}_' + enc.categories_[0])
    new_cols.index = Train.index
    Train = pd.concat([Train, new_cols], axis = 1)
    Train = Train.drop(col, axis = 1)
    
for log_col in train_scale.columns[train_scale.columns.str.endswith('log')]:
    col = log_col.split('_log')[0]
    Train[log_col] = np.log(Train[col])
    Train = Train.drop(col, axis = 1)
    
Train = Train[train_scale.columns]

#Preprocess test data
for model_file in listing:
    col = model_file.split('_one_')[0]

    col = col.split('/')[-1]
    enc = pickle.load(open(model_file, 'rb'))
    new_cols = pd.DataFrame(enc.transform(Test[[col]]).toarray(),
                    columns=f'{col}_' + enc.categories_[0])
    
    new_cols.index = Test.index
    Test = pd.concat([Test, new_cols], axis = 1)
    Test = Test.drop(col, axis = 1)
    
for log_col in test_scale.columns[test_scale.columns.str.endswith('log')]:
    col = log_col.split('_log')[0]
    Test[log_col] = np.log(Test[col])
    Test = Test.drop(col, axis = 1)
    
Test = Test[test_scale.columns]

In [3]:
# for col in Test.columns:
#     if len(set(Train[col])) > 2 and '_log' not in col:
#         Train[f'{col}_log'] = np.log(Train[col] + 1)
#         Train = Train.drop(col, axis = 1)
        
#         Test[f'{col}_log'] = np.log(Test[col] + 1)
#         Test = Test.drop(col, axis = 1)

In [4]:
# Train['GrLivArea_log_square'] = Train['GrLivArea_log']**2
# Test['GrLivArea_log_square'] = Test['GrLivArea_log']**2
# for col in Train.columns:
#     if '_log' in col and col != 'SalePrice_log':
#         Train[col + '_square'] = Train[col] **2

# for col in Test.columns:
#     if '_log' in col and col != 'SalePrice_log':
#         Test[col + '_square'] = Test[col] **2

# Batch fold spliter

In [5]:
def getFoldValidation(length, K = 4):
    diff = int(length/K)
    fraction = length % K
    batchValidation = [[k*diff, (k + 1)*diff] for k in range(K)]
    batchValidation[-1][1] = batchValidation[-1][1] + fraction - 1
    return batchValidation
batchValidation = np.array(getFoldValidation(len(Train)))

In [6]:
def getTrainTestBatch():
    fold = 0
    while fold < batchValidation.shape[0]:
        train_batch = pd.concat([Train.iloc[batchValidation[i][0]:batchValidation[i][1]]
             for i in range(batchValidation.shape[0]) if i !=  fold])
        test_batch = Train.iloc[batchValidation[fold][0]:batchValidation[fold][1]]
        yield train_batch, test_batch, fold
        fold += 1
train_batch, test_batch, fold = next(getTrainTestBatch())     
                                

# Stack Model

* based on https://www.kaggle.com/serigne/stacked-regressions-top-4-on-leaderboard#Modelling

## import modules

In [7]:
from sklearn.linear_model import ElasticNet, Lasso,  BayesianRidge, LassoLarsIC
from sklearn.ensemble import RandomForestRegressor,  GradientBoostingRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import lightgbm as lgb

In [8]:
import keras.backend as K

def mse_loss(y_true, y_pred):
    return K.sqrt(K.mean((y_true - y_pred)**2))
def e_loss(y_true, y_pred):
    return K.sqrt(K.mean((np.e**y_true - np.e**y_pred)**2))


In [9]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras.layers import Input
from tensorflow.keras.layers.experimental import preprocessing
from tensorflow_addons.layers import WeightNormalization
from keras.callbacks import ModelCheckpoint,EarlyStopping
from tensorflow.keras import regularizers
from tensorflow.keras import metrics

from keras.layers import Dropout

from sklearn.tree import DecisionTreeRegressor
from sklearn.isotonic import IsotonicRegression

genarator =  getTrainTestBatch()




## Base model

### Lasso

In [10]:
lasso = make_pipeline(RobustScaler() #Scaler for make the model more robust to outlier
                      ,Lasso(alpha =0.0005, random_state=1))

### Elastic Net Regression

In [11]:
ENet = make_pipeline(RobustScaler(),
                     ElasticNet(alpha=0.0005, l1_ratio=.9, random_state=3))

### Kernel Ridge Regression 

In [12]:
KRR = KernelRidge(alpha=0.6, kernel='polynomial', degree=2, coef0=2.5)

### Gradient Boosting Regressor

In [13]:
GBoost = GradientBoostingRegressor(n_estimators=3000, learning_rate=0.05,
                                   max_depth=4, max_features='sqrt',
                                   min_samples_leaf=15, min_samples_split=10, 
                                   loss='huber', random_state =5)

### XGBoost

In [14]:
model_xgb = xgb.XGBRegressor(colsample_bytree=0.4603, gamma=0.0468, 
                             learning_rate=0.05, max_depth=3, 
                             min_child_weight=1.7817, n_estimators=2200,
                             reg_alpha=0.4640, reg_lambda=0.8571,
                             subsample=0.5213, silent=1,
                             random_state =7, nthread = -1)

### LightGBM

In [15]:
model_lgb = lgb.LGBMRegressor(objective='regression',num_leaves=5,
                              learning_rate=0.05, n_estimators=720,
                              max_bin = 55, bagging_fraction = 0.8,
                              bagging_freq = 5, feature_fraction = 0.2319,
                              feature_fraction_seed=9, bagging_seed=9,
                              min_data_in_leaf =6, min_sum_hessian_in_leaf = 11)

### Validation function

In [16]:
Train

,MSSubClass,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,1stFlrSF,2ndFlrSF,LowQualFinSF,FullBath,...,PavedDrive_N,PavedDrive_P,PavedDrive_Y,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial,SalePrice_log
Id,,,,,,,,,,,,,,,,,,,,,
1,60,8450,7,5,2003,2003,856,854,0,2,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,12.247694
2,20,9600,6,8,1976,1976,1262,0,0,2,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,12.109011
3,60,11250,7,5,2001,2002,920,866,0,2,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,12.317167
4,70,9550,7,5,1915,1970,961,756,0,1,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,11.849398
5,60,14260,8,5,2000,2000,1145,1053,0,2,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,12.429216
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,60,7917,6,5,1999,2000,953,694,0,2,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,12.072541
1457,20,13175,6,6,1978,1988,2073,0,0,2,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,12.254863
1458,70,9042,7,9,1941,2006,1188,1152,0,2,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,12.493130


In [17]:
X_Train = Train.drop('SalePrice_log', axis = 1)
X_Train

,MSSubClass,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,1stFlrSF,2ndFlrSF,LowQualFinSF,FullBath,...,Electrical_SBrkr,PavedDrive_N,PavedDrive_P,PavedDrive_Y,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
Id,,,,,,,,,,,,,,,,,,,,,
1,60,8450,7,5,2003,2003,856,854,0,2,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,20,9600,6,8,1976,1976,1262,0,0,2,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,60,11250,7,5,2001,2002,920,866,0,2,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,70,9550,7,5,1915,1970,961,756,0,1,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
5,60,14260,8,5,2000,2000,1145,1053,0,2,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,60,7917,6,5,1999,2000,953,694,0,2,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1457,20,13175,6,6,1978,1988,2073,0,0,2,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1458,70,9042,7,9,1941,2006,1188,1152,0,2,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [18]:
#Validation function


def rmsle_cv(model, n_folds = 5):
    X_Train = Train.drop('SalePrice_log', axis = 1).values
    Y_Train = Train[['SalePrice_log']].values
    kf = (KFold(n_folds, shuffle=True,
                random_state=42
               ).
              get_n_splits(X_Train))
    rmse= np.sqrt(-cross_val_score(model, X_Train, Y_Train, scoring="neg_mean_squared_error", cv = kf))
    return(rmse)
score = rmsle_cv(lasso)
print("\nLasso score: {:.4f} ({:.4f})\n".format(score.mean(), score.std()))

score = rmsle_cv(ENet)
print("ElasticNet score: {:.4f} ({:.4f})\n".format(score.mean(), score.std()))

score = rmsle_cv(KRR)
print("Kernel Ridge score: {:.4f} ({:.4f})\n".format(score.mean(), score.std()))

# score = rmsle_cv(GBoost)
# print("Gradient Boosting score: {:.4f} ({:.4f})\n".format(score.mean(), score.std()))

# score = rmsle_cv(model_xgb)
# print("Xgboost score: {:.4f} ({:.4f})\n".format(score.mean(), score.std()))

# score = rmsle_cv(model_lgb)
# print("LGBM score: {:.4f} ({:.4f})\n" .format(score.mean(), score.std()))


Lasso score: 0.1272 (0.0064)

ElasticNet score: 0.1272 (0.0064)

Kernel Ridge score: 0.8795 (0.6566)



## Stacking models

### Averaging base models

In [19]:
class AveragingModels(BaseEstimator, RegressorMixin, TransformerMixin):
    def __init__(self, models):
        self.models = models
        
    # we define clones of the original models to fit the data in
    def fit(self, X, y):
        self.models_ = [clone(x) for x in self.models]
        
        # Train cloned base models
        for model in self.models_:
            model.fit(X, y)

        return self
    
    #Now we do the predictions for cloned models and average them
    def predict(self, X):
        predictions = np.column_stack([
            model.predict(X) for model in self.models_
        ])
        return np.mean(predictions, axis=1)

averaged_models = AveragingModels(models = [ENet, GBoost, KRR, lasso])

score = rmsle_cv(averaged_models)
print(" Averaged base models score: {:.4f} ({:.4f})\n".format(score.mean(), score.std()))

 Averaged base models score: 0.2578 (0.1512)



### Add meta model 

In [20]:
class StackingAveragedModels(BaseEstimator, RegressorMixin, TransformerMixin):
    def __init__(self, base_models, meta_model, n_folds=5):
        self.base_models = base_models
        self.meta_model = meta_model
        self.n_folds = n_folds
   
    # We again fit the data on clones of the original models
    def fit(self, X, y):
        self.base_models_ = [list() for x in self.base_models]
        self.meta_model_ = clone(self.meta_model)
        kfold = KFold(n_splits=self.n_folds, shuffle=True, random_state=156)
        
        # Train cloned base models then create out-of-fold predictions
        # that are needed to train the cloned meta-model
        out_of_fold_predictions = np.zeros((X.shape[0], len(self.base_models)))
        for i, model in enumerate(self.base_models):
            for train_index, holdout_index in kfold.split(X, y):
                
                instance = clone(model)
                
                self.base_models_[i].append(instance)
                
                instance.fit(X[train_index], y[train_index])
                
                y_pred = instance.predict(X[holdout_index])
                
                out_of_fold_predictions[holdout_index, i] = y_pred.flatten()
                
        # Now train the cloned  meta-model using the out-of-fold predictions as new feature
        self.meta_model_.fit(out_of_fold_predictions, y)
        return self
   
    #Do the predictions of all base models on the test data and use the averaged predictions as 
    #meta-features for the final prediction which is done by the meta-model
    def predict(self, X):
        meta_features = np.column_stack([
            np.column_stack([model.predict(X) for model in base_models]).mean(axis=1)
            for base_models in self.base_models_ ])
        return self.meta_model_.predict(meta_features)
    
stacked_averaged_models = StackingAveragedModels(base_models = (ENet, GBoost, KRR),
                                                 meta_model = LinearRegression())

score = rmsle_cv(stacked_averaged_models)
print("Stacking Averaged models score: {:.4f} ({:.4f})".format(score.mean(), score.std()))

Stacking Averaged models score: 0.1216 (0.0067)


In [21]:
X_Train = Train.drop('SalePrice_log', axis = 1)
Y_Train = Train[['SalePrice_log']]

stacked_averaged_models.fit(X_Train.values, Y_Train.values)

StackingAveragedModels(base_models=(Pipeline(steps=[('robustscaler',
                                                     RobustScaler()),
                                                    ('elasticnet',
                                                     ElasticNet(alpha=0.0005,
                                                                l1_ratio=0.9,
                                                                random_state=3))]),
                                    GradientBoostingRegressor(learning_rate=0.05,
                                                              loss='huber',
                                                              max_depth=4,
                                                              max_features='sqrt',
                                                              min_samples_leaf=15,
                                                              min_samples_split=10,
                                                              n_estimat

In [22]:
stacked_averaged_models = StackingAveragedModels(base_models = (ENet, GBoost, KRR),
                                                 meta_model = lasso)

score = rmsle_cv(stacked_averaged_models)
print("Stacking Averaged models score: {:.4f} ({:.4f})".format(score.mean(), score.std()))

Stacking Averaged models score: 0.1216 (0.0068)


In [23]:
def rmsle(y, y_pred):
    return np.sqrt(mean_squared_error(y, y_pred))

In [24]:
stacked_averaged_models.fit(Train.drop('SalePrice_log', axis = 1).values, Train[['SalePrice_log']].values)
stacked_train_pred = stacked_averaged_models.predict(Train.drop('SalePrice_log', axis = 1).values)

In [25]:
print(rmsle(Train[['SalePrice_log']].values, stacked_train_pred))

0.08812587971271589


In [26]:
predict = stacked_averaged_models.predict(Test.values)

# Postprocessing

In [27]:
predict = pd.DataFrame(predict)
predict.index = Test.index
predict.columns = ['SalePrice_log']
predict['SalePrice'] = np.e**(predict['SalePrice_log'])
predict = predict.drop('SalePrice_log', axis = 1)
predict

,SalePrice
Id,
1461,122223.608731
1462,154640.308441
1463,177693.960831
1464,194332.529808
1465,206458.333388
...,...
2915,93160.765870
2916,81649.906742
2917,150500.245836


In [28]:
predict.to_csv("submission.csv")